# ⚡ Playground 1: 2D Point GAN Arena (Cơ Chế Đối Kháng Cốt Lõi)
### Trực quan hóa cuộc đấu giữa "Kẻ Làm Giả" và "Cảnh Sát" trên mặt phẳng 2D

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDanh1510/GAN-playground/blob/main/notebooks/01_2d_point_gan_playground.ipynb)

---

## 🎯 Mục Tiêu Bài Học:
1. Hiểu cách **Generator ($G$)** biến đổi nhiễu ngẫu nhiên Gauss $z \sim \mathcal{N}(0, 1)$ thành hình dạng mục tiêu (Hình Trái Tim 💖, Vòng Tròn ⭕, Đường Xoắn Ốc 🌀).
2. Hiểu cách **Discriminator ($D$)** tạo ra đường biên phân loại (*Decision Boundary*) để phân biệt điểm thật vs điểm giả.
3. Tận mắt quan sát sự cân bằng Nash (*Nash Equilibrium*) và cuộc chiến mất mát (*Loss Battle*).

### 1. Cài đặt môi trường & Kiểm tra thiết bị

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Đang sử dụng thiết bị: {device}")
if torch.cuda.is_available():
    print(f"🔥 Tên GPU: {torch.cuda.get_device_name(0)}")

### 2. Định nghĩa Kiến trúc Mạng Nơ-ron:
- **Generator**: Đầu vào $z \in \mathbb{R}^2 \to$ Hidden Layers $\to$ Tọa độ $(x, y) \in \mathbb{R}^2$
- **Discriminator**: Tọa độ $(x, y) \in \mathbb{R}^2 \to$ Hidden Layers $\to$ Xác suất $P(\text{Real}) \in [0, 1]$

In [ ]:
class Generator2D(nn.Module):
    def __init__(self, latent_dim=2, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 2)
        )
    def forward(self, z):
        return self.net(z)

class Discriminator2D(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.2),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

print("✓ Khởi tạo Generator & Discriminator thành công!")

### 3. Bộ tạo dữ liệu phân phối điểm mẫu (Hình Trái Tim 💖)

In [ ]:
def get_heart_distribution(batch_size=256):
    t = torch.rand(batch_size) * 2 * np.pi - np.pi
    x = 16 * (torch.sin(t) ** 3) / 18
    y = (13 * torch.cos(t) - 5 * torch.cos(2*t) - 2 * torch.cos(3*t) - torch.cos(4*t)) / 18
    noise = torch.randn(batch_size, 2) * 0.04
    return torch.stack([x, y], dim=1) + noise

# Vẽ thử phân phối dữ liệu thật
sample_data = get_heart_distribution(500).numpy()
plt.figure(figsize=(5, 5))
plt.scatter(sample_data[:, 0], sample_data[:, 1], c='#f97316', s=15, alpha=0.7)
plt.title('Dữ liệu Thật: Hình Trái Tim')
plt.grid(True, alpha=0.3)
plt.show()

### 4. Vòng lặp Huấn Luyện GAN & Trực Quan Hóa Quá Trình Học

In [ ]:
epochs = 300
batch_size = 256
lr = 0.005

G = Generator2D().to(device)
D = Discriminator2D().to(device)
criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

loss_history = {'d': [], 'g': []}
snapshots = {}

print("⚡ Đang bắt đầu huấn luyện 2D GAN...")
for epoch in range(1, epochs + 1):
    # --- BƯỚC 1: TRAIN DISCRIMINATOR ---
    real_pts = get_heart_distribution(batch_size).to(device)
    noise = torch.randn(batch_size, 2, device=device)
    fake_pts = G(noise)
    
    loss_real = criterion(D(real_pts), torch.ones(batch_size, 1, device=device) * 0.9) # Label smoothing
    loss_fake = criterion(D(fake_pts.detach()), torch.zeros(batch_size, 1, device=device))
    d_loss = (loss_real + loss_fake) / 2
    
    opt_D.zero_grad(); d_loss.backward(); opt_D.step()
    
    # --- BƯỚC 2: TRAIN GENERATOR ---
    g_loss = criterion(D(fake_pts), torch.ones(batch_size, 1, device=device))
    opt_G.zero_grad(); g_loss.backward(); opt_G.step()
    
    loss_history['d'].append(d_loss.item())
    loss_history['g'].append(g_loss.item())
    
    if epoch in [1, 50, 150, 300]:
        with torch.no_grad():
            snapshots[epoch] = G(torch.randn(350, 2, device=device)).cpu().numpy()
        print(f"Epoch [{epoch:03d}/{epochs}] | D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")

print("✓ Huấn luyện hoàn tất!")

### 5. Xem diễn biến quá trình biến đổi qua các Epoch

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
real_ref = get_heart_distribution(350).numpy()

for idx, (ep, pts) in enumerate(snapshots.items()):
    ax = axes[idx]
    ax.scatter(real_ref[:, 0], real_ref[:, 1], c='#f97316', s=10, alpha=0.3, label='Thật')
    ax.scatter(pts[:, 0], pts[:, 1], c='#06b6d4', s=14, alpha=0.8, label='Generator')
    ax.set_title(f"Epoch {ep}")
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
    ax.grid(True, alpha=0.2)
    if idx == 0: ax.legend()

plt.suptitle("Sự Tiến Hóa Của Generator Từ Nhiễu Loạn Đến Hình Trái Tim", fontsize=14, fontweight='bold')
plt.show()

### 6. Biểu đồ cuộc chiến Loss (Discriminator vs Generator)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(loss_history['d'], label='D Loss (Cảnh sát)', color='#f97316')
plt.plot(loss_history['g'], label='G Loss (Kẻ làm giả)', color='#06b6d4')
plt.title('Biểu Đồ Loss Trận Đấu GAN (D Loss vs G Loss Battle)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 💡 Câu Hỏi Thử Thách Cho Học Sinh:
1. Điều gì xảy ra nếu bạn tăng tốc độ học của Discriminator lên gấp 10 lần ($lr_D = 0.05$ trong khi $lr_G = 0.005$)?
2. Hãy thử thay đổi hàm sinh dữ liệu `get_heart_distribution` thành phương trình đường tròn $x^2 + y^2 = r^2$ để xem Generator học ra sao!